In [41]:
import pandas as pd
from functools import reduce
import ast
from pathlib import Path
from config import RESULTS_DIR

In [2]:
# some helpers
def parse_value(x):
    if isinstance(x, str) and x.startswith("["): # sometimes I saved values in list, likely from LDDMM output
        return ast.literal_eval(x)[0]
    return float(x)

In [ ]:
fname = RESULTS_DIR / "metrics" / "version_0-LDDMM-trainshapes.parquet"
df = pd.read_parquet(fname)
df["value"] = df["value"].apply(parse_value)
df

Mean error per version, per organ, across patients

In [ ]:
df.groupby(["version", "metric", "organ"])["value"].agg(
    mean="mean",
    median="median",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

Mean error per version, across organs, across patients : one single number per version. Organs are weighted the same here.

In [ ]:
df.groupby(["version", "metric"])["value"].agg(
    mean="mean",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

# Load several versions / metrics

In [21]:
# retrieve all the wanted files
dfs = []

for file_path in Path("results/metrics").glob("version_*.parquet"):
    # extract version from filename, e.g. version_89.csv → 89
    version = file_path.stem.split("-")[0].split("_")[-1]
    df = pd.read_parquet(file_path)
    df["version"] = int(version)
    df["value"] = df["value"].apply(parse_value)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
# df_across_vers["version"].unique() # inspect which versions are there
df_all

/tmp/ipykernel_3925478/3287992365.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(x)
/tmp/ipykernel_3925478/3287992365.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(x)
/tmp/ipykernel_3925478/3287992365.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(x)


,version,patient,organ,metric,value
0,114,AF059,epicardium,LDDMM,0.012995
1,114,AF059,la_endo,LDDMM,0.012610
2,114,AF059,ra_endo,LDDMM,0.001626
3,114,LEU_NORM_F017,epicardium,LDDMM,0.004664
4,114,LEU_NORM_F017,la_endo,LDDMM,0.005728
...,...,...,...,...,...
175,100,AF037,la_endo,LDDMM,1.023166
176,100,AF037,ra_endo,LDDMM,0.016999
177,100,LEU_NORM_0717,epicardium,LDDMM,0.038645
178,100,LEU_NORM_0717,la_endo,LDDMM,0.006885


In [22]:
df_all.groupby(["version", "metric", "organ"])["value"].agg(
    mean="mean",
    median="median",
    std="std",
    q95=lambda x: x.quantile(0.95),
)

mean    median       std       q95
version metric  organ                                             
89      LDDMM   epicardium  0.050874  0.034255  0.053713  0.136110
                la_endo     0.124234  0.020904  0.328165  0.604930
                ra_endo     0.017285  0.012245  0.015116  0.043434
        chamfer epicardium  0.038532  0.034915  0.012231  0.057736
                la_endo     0.050239  0.042567  0.015669  0.076686
                ra_endo     0.049449  0.042124  0.015545  0.072878
100     LDDMM   epicardium  0.074344  0.064056  0.059302  0.165759
                la_endo     0.111134  0.007143  0.320532  0.575388
                ra_endo     0.015760  0.008786  0.015041  0.041536
        chamfer epicardium  0.039617  0.040101  0.009228  0.052566
                la_endo     0.048519  0.050083  0.009943  0.060854
                ra_endo     0.052022  0.054606  0.012100  0.068549
114     LDDMM   epicardium  0.022527  0.010189  0.025852  0.065143
                la_endo     0.114323  0.009169  0.313919  0.578367
                ra_endo     0.017210  0.002941  0.029987  0.072074
        chamfer epicardium  0.027821  0.027305  0.010946  0.040410
                la_endo     0.037169  0.038756  0.014380  0.056875
                ra_endo     0.041515  0.032983  0.027775  0.085856

# Ranking versions

In [29]:
df_chamfer_epi = df_all.query(" metric == 'chamfer' and organ == 'epicardium' ")
df_LDDMM_epi = df_all.query(" metric == 'LDDMM' and organ == 'epicardium' ")
df_chamfer_la = df_all.query(" metric == 'chamfer' and organ == 'la_endo' ")
df_LDDMM_la = df_all.query(" metric == 'LDDMM' and organ == 'la_endo' ")
df_chamfer_ra = df_all.query(" metric == 'chamfer' and organ == 'ra_endo' ")
df_LDDMM_ra = df_all.query(" metric == 'LDDMM' and organ == 'ra_endo' ")

Using vectorized built-in rank function of pandas. Leaving metric and organ columns for clarity and to not loose information of what this data is from.

In [38]:
ranked_chamfer = []
col_names = ["mean_rank_epi", "mean_rank_la", "mean_rank_ra"]
for i,df in enumerate([df_chamfer_epi, df_chamfer_la, df_chamfer_ra]):

    df_ranked = df.copy()
    df_ranked["rank"] = (
        df_ranked
        .groupby("patient")["value"]
        .rank(method="min", ascending=True)
    )
    df_ranked = df_ranked.groupby(["version"])["rank"].agg(mean="mean").reset_index()
    df_ranked = df_ranked.rename(columns={"mean": col_names[i]})
    ranked_chamfer.append(df_ranked)


In [43]:
df_ranked_all = reduce( lambda left, right: pd.merge(left, right, on="version", how="inner"), ranked_chamfer )
df_ranked_all

,version,mean_rank_epi,mean_rank_la,mean_rank_ra
0,89,2.5,2.3,2.3
1,100,2.4,2.4,2.4
2,114,1.1,1.3,1.3


In [46]:
df_long = df_ranked_all.melt(
    id_vars="version",        # keep version
    value_vars=["mean_rank_epi", "mean_rank_la", "mean_rank_ra"],  # columns to stack
    var_name="organ",         # name for the new “column identifier”
    value_name="mean_rank"    # name for the values
)
df_long.groupby("version")["mean_rank"].mean().reset_index()

,version,mean_rank
0,89,2.366667
1,100,2.400000
2,114,1.233333
